In [ ]:
cd /g/data/jk72/deg581/seqom/analysis/notebooks

In [ ]:
%%time
# load modules
## Data processing and DA modules
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, zoomed_inset_axes
import matplotlib.ticker as mticker

## Dealing with big data and netcdf
import xarray as xr
from netCDF4 import Dataset
## ROMS packages
from xgcm import Grid
## color maps
import cmaps
import cmocean
## mapping packages
import cartopy.crs as ccrs
import cartopy.feature as cfeature
## System tools and python configuration
import os
import glob
# import repackage
# repackage.add('../../')
# repackage.add('../')

from scipy.ndimage import gaussian_filter1d


Define functions

In [ ]:

def inpolygon(xq, yq, xv, yv):
    from matplotlib import path
    shape = xq.shape
    xq = xq.reshape(-1)
    yq = yq.reshape(-1)
    xv = xv.reshape(-1)
    yv = yv.reshape(-1)
    q = [(xq[i], yq[i]) for i in range(xq.shape[0])]
    p = path.Path([(xv[i], yv[i]) for i in range(xv.shape[0])])
    return p.contains_points(q).reshape(shape)

# map u,v to rho points
def ROMSmetricsAndGrid(ds):
    ds = ds.rename({'eta_u': 'eta_rho', 'xi_v': 'xi_rho', 'xi_psi': 'xi_u', 'eta_psi': 'eta_v'})

    coords={'X':{'center':'xi_rho', 'inner':'xi_u'}, 
        'Y':{'center':'eta_rho', 'inner':'eta_v'}, 
        'Z':{'center':'s_rho', 'outer':'s_w'}}

    grid = Grid(ds, coords=coords, periodic=[])

    print('making pm/pn metrics')
    ds['pm_v'] = grid.interp(ds.pm, 'Y')
    ds['pn_u'] = grid.interp(ds.pn, 'X')
    ds['pm_u'] = grid.interp(ds.pm, 'X')
    ds['pn_v'] = grid.interp(ds.pn, 'Y')
    ds['pm_psi'] = grid.interp(grid.interp(ds.pm, 'Y'),  'X') # at psi points (eta_v, xi_u) 
    ds['pn_psi'] = grid.interp(grid.interp(ds.pn, 'X'),  'Y') # at psi points (eta_v, xi_u)
    print('making dx/dy')
    ds['dx'] = 1/ds.pm
    ds['dx_u'] = 1/ds.pm_u
    ds['dx_v'] = 1/ds.pm_v
    ds['dx_psi'] = 1/ds.pm_psi

    ds['dy'] = 1/ds.pn
    ds['dy_u'] = 1/ds.pn_u
    ds['dy_v'] = 1/ds.pn_v
    ds['dy_psi'] = 1/ds.pn_psi

#     ds['dz'] = grid.diff(ds.z_w, 'Z', boundary='fill')
#     ds['dz_w'] = grid.diff(ds.z_rho, 'Z', boundary='fill')
#     ds['dz_u'] = grid.interp(ds.dz, 'X')
#     ds['dz_w_u'] = grid.interp(ds.dz_w, 'X')
#     ds['dz_v'] = grid.interp(ds.dz, 'Y')
#     ds['dz_w_v'] = grid.interp(ds.dz_w, 'Y')

    ds['dA'] = ds.dx * ds.dy

    metrics = {
        ('X',): ['dx', 'dx_u', 'dx_v', 'dx_psi'], # X distances
        ('Y',): ['dy', 'dy_u', 'dy_v', 'dy_psi'], # Y distances
        # ('Z',): ['dz', 'dz_u', 'dz_v', 'dz_w', 'dz_w_u', 'dz_w_v'], # Z distances
        ('X', 'Y'): ['dA'] # Areas
    }
    grid = Grid(ds, coords=coords, metrics=metrics, periodic=[])

    return ds,grid



def add_zeros_to_4(date):
    if date<10:
        to_add = '000'
    elif date>9 & date<100:
        to_add = '00'
    elif date>99 & date < 1000:
        to_add = '0'
    else: 
        to_add = ''
    return to_add

def generateFileList(FilePath,prefix,datelist):
    filelist=[FilePath+prefix+add_zeros_to_4(datelist[0])+str(datelist[0])+'.nc']
    for dates in datelist[1:]:
        filenameToAppend=FilePath+prefix+add_zeros_to_4(dates)+str(dates)+'.nc'
        filelist.append(filenameToAppend)
    return filelist

def vorticity(u, v, grd):
    """
    compute the relative vorticity
    https://github.com/ESMG/pyroms/blob/python3/pyroms_toolbox/pyroms_toolbox/vorticity.py
    e.g. vorticity(ds_CTRL.u.isel(s_rho=-1,ocean_time=1),ds_CTRL.v.isel(s_rho=-1,ocean_time=1),grd)
    """

    dx = 1./grd.pm.values
    dy = 1./grd.pn.values

    #dx, dy at psi point
    dx = 0.5 * (dx[1:,:] + dx[:-1,:])
    dy = 0.5 * (dy[:,1:] + dy[:,:-1])

    vorticity = np.zeros(grd.mask_psi.shape)

    vorticity = 2 * (v[:,1:] - v[:,:-1]) / (dx[:,1:] + dx[:,:-1]) \
                - 2 * (u[1:,:] - u[:-1,:]) / (dy[1:,:] + dy[:-1,:])

    vorticity = np.ma.masked_where(grd.mask_psi == 0, vorticity)

def N2(rho, z, rho_0=1000.0):
    '''
    Return the stratification frequency
    
    Parameters
    ----------
    rho : array_like
        density [kg/m^3]
    z : array_like
        depths [m] (positive upward)
    
    Returns
    -------
    N2 : array_like
        Stratification frequency [1/s], where the vertical dimension (the
        first dimension) is one less than the input arrays.    
    '''
    rho = np.asarray(rho)
    z = np.asarray(z)
    assert rho.shape == z.shape, 'rho and z must have the same shape.'
    r_z = np.diff(rho, axis=0) / np.diff(z, axis=0)
    return -(9.8 / rho_0) * r_z



Load grid

In [ ]:
grd = xr.open_dataset('/g/data/jk72/deg581/se-qld-setup/data/proc/seqld_1km_v1.7_grd.nc')


Load data file

In [ ]:


# FilePath='/g/data/jk72/deg581/seqom/seqld_2012/' # Truth file settings
FilePath='/g/data/jk72/deg581/seqom/seqom_v1.7_2012/' # Truth file settings
prefix='roms_his_'
timeRange = [12, 13]
datelist = np.array(range(timeRange[0],timeRange[1],1))


fl=generateFileList(FilePath,prefix,datelist)
print(fl)

# ds=loadOverlappedNetcdfFileList(filelist=fl,overlapDays=7)

ds = xr.open_mfdataset(fl,chunks = {'ocean_time':1}, data_vars='minimal', compat='override',coords='minimal',parallel='False',join='right')

print(ds.nbytes/1e9,'G')

ds = ds.drop_vars(['sustr','svstr','ssflux','shflux','ubar_eastward','vbar_northward','w',])
print(ds.nbytes/1e9,'G')
ds



ds = ds.assign_coords({"lon_rho": grd.lon_rho})
ds = ds.assign_coords({"lat_rho": grd.lat_rho})

# ds['hc'] = grd.hc
# ds['s_w']=grd.s_w
# ds['s_rho']=grd.s_rho
# ds['Cs_w']=grd.Cs_w
# ds['Cs_r']=grd.Cs_r


# ds_raw = ds_raw.assign_coords(lon_rho=grd.lon_rho)
# ds_raw = ds_raw.assign_coords(lat_rho=grd.lat_rho)


ds['mask_rhoNaN'] = ds.mask_rho.where(ds.mask_rho)

# make masks

# poly_shelf = np.array([
#     [1.55e6,600000],
#     [2.25e6,600000],
#     [2.25e6,800000],
#     [2.20e6,830000],
#     [1.55e6,890000]])
# # plt.plot(poly_shelf[:,0],poly_shelf[:,1])
# # plt.show()



# mask_roi = inpolygon(ds.x_rho.values, ds.y_rho.values,poly_shelf[:,0], poly_shelf[:,1])


# ds['mask_zice_roi'] = ds.mask_zice*mask_roi
# roi_label1 = 'amery'
# ds.mask_zice_roi.attrs['long_name']=roi_label1

# set any grid data here.

weights_area = (1/ds.pm)*(1/ds.pn)
weights_area.name = "weights"

print('making vertical coordinates')
Zo_rho = (ds.hc * ds.s_rho + ds.Cs_r * ds.h) / (ds.hc + ds.h)
z_rho = ( ds.h) * Zo_rho
Zo_w = (ds.hc * ds.s_w + ds.Cs_w * ds.h) / (ds.hc + ds.h)
z_w = Zo_w * ( ds.h) 
    
ds.coords['z_w0'] = z_w.where(ds.mask_rho, 0).transpose('s_w', 'eta_rho', 'xi_rho')
ds.coords['z_rho0'] = z_rho.where(ds.mask_rho, 0).transpose('s_rho', 'eta_rho', 'xi_rho')

ds['dz'] = (('s_rho', 'eta_rho', 'xi_rho'),np.diff(ds.z_w0,axis=0))


ds, grid = ROMSmetricsAndGrid(ds)

# print('mapping u/v to u/v rho')
# ds['u_rho'] = grid.interp(ds.u,'X')
# ds['v_rho'] = grid.interp(ds.v,'Y')

ds_CTRL = ds

ds.close()

Define time periods

In [ ]:
winter_period=slice(30,40)
summer_period=slice(72,73)

winter_period_snapshot=slice(38,39)
summer_period_snapshot=slice(72,73)

In [ ]:
# from scipy.stats import skew
# zeta_skew_map = xr.apply_ufunc(
#     skew,
#     rel_vort_norm,
#     input_core_dims=[['ocean_time']],
#     vectorize=True,
#     kwargs={"nan_policy": "omit"}
# )

Do any analyses

In [ ]:
# potential vorticity
rel_vort = grid.derivative(ds_CTRL.v, 'X') - grid.derivative(ds_CTRL.u, 'Y')
rel_vort_norm = rel_vort / grid.interp(grid.interp(ds_CTRL.f,'X'),'Y')

# strain = np.sqrt( (grid.derivative(ds_CTRL.u,'X') - grid.derivative(ds_CTRL.v,'Y'))**2 + (grid.derivative(ds_CTRL.u,'Y') + grid.derivative(ds_CTRL.v,'X'))**2 )
# strain_norm = strain / grid.interp(grid.interp(ds_CTRL.f,'X'),'Y')

div = grid.derivative(ds_CTRL.u, 'X') + grid.derivative(ds_CTRL.v, 'Y')
div_norm = div / ds_CTRL.f

#skew

# skew_period = slice(25,40)
skew_period = slice(0,73)


rel_vort.load()
from scipy.stats import skew
rel_vort.load()
rel_vort_subset= rel_vort.isel(s_rho=-1).isel(ocean_time=skew_period)
ntime, ny, nx = rel_vort_subset.shape
zeta_skew_array = np.full((ny, nx), np.nan)# Preallocate result array
for j in range(ny): # Loop over each grid point
    print(j)
    for i in range(nx):
        series = rel_vort_subset[:, j, i]
        if np.all(np.isnan(series)):
            continue
        zeta_skew_array[j, i] = skew(series, nan_policy='omit')
# zeta_skew_map = xr.DataArray(
#     zeta_skew_array,
#     dims=('y', 'x'),
#     coords={'y': rel_vort['eta_rho'], 'x': rel_vort['xi_rho']},
#     name='zeta_skew'
# )
zeta_skew = zeta_skew_array

#mke
u_bar = ds_CTRL.u_eastward.mean("ocean_time")
v_bar = ds_CTRL.v_northward.mean("ocean_time")
mke = 0.5*(u_bar**2 + v_bar**2)



print('calc velocity anomalies')
u_prime = ds.u_eastward - u_bar
v_prime = ds.v_northward - v_bar

print('calc eke')
eke = 0.5*(u_prime**2 + v_prime**2)



#eke

In [ ]:


plt.cla()
plt.clf()
fig = plt.figure(figsize=[15,10])
ax = None

gs = gridspec.GridSpec(nrows=1,ncols=2,wspace=0.035, hspace=0.035)

choose_depth=0

ax=fig.add_subplot(gs[0,0])
im = ax.pcolormesh(grd.lon_rho,grd.lat_rho,rel_vort_norm.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),cmap='RdBu_r',vmin=-3,vmax=3)
ax.contour(grd.lon_psi,grd.lat_psi,rel_vort_norm.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),colors='k',linewidths=.5)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.set_xlim((153,156))


cax = inset_axes(ax,
                width="2%",  # width = 10% of parent_bbox width
                height="10%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.01,0.1, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='vertical') 
cax.set_ylabel(r' Normalised rel. vorticity / divergence')#,fontsize=14)

# ax=fig.add_subplot(gs[0,1])
# ax.pcolormesh(grd.lon_rho,grd.lat_rho,strain_norm.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),cmap='RdBu_r',vmin=-4,vmax=4)
# ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
# ax.set_xlim((153,156))

ax=fig.add_subplot(gs[0,1])
ax.pcolormesh(grd.lon_rho,grd.lat_rho,div_norm.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),cmap='RdBu_r',vmin=-3,vmax=3)
ax.contour(grd.lon_rho,grd.lat_rho,div_norm.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),colors='k',linewidths=.5)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.set_xlim((153,156))





In [ ]:


plt.cla()
plt.clf()
fig = plt.figure(figsize=[15,10])
ax = None

gs = gridspec.GridSpec(nrows=1,ncols=2,wspace=0.035, hspace=0.035)

choose_depth=0

ax=fig.add_subplot(gs[0,0])
im = ax.pcolormesh(grd.lon_rho,grd.lat_rho,rel_vort_norm.isel(s_rho=-1).isel(ocean_time=summer_period_snapshot).squeeze(),cmap='RdBu_r',vmin=-3,vmax=3)
ax.contour(grd.lon_psi,grd.lat_psi,rel_vort_norm.isel(s_rho=-1).isel(ocean_time=summer_period_snapshot).squeeze(),colors='k',linewidths=.5)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.set_xlim((153,156))


cax = inset_axes(ax,
                width="2%",  # width = 10% of parent_bbox width
                height="10%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.01,0.1, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='vertical') 
cax.set_ylabel(r' Normalised rel. vorticity / divergence')#,fontsize=14)

# ax=fig.add_subplot(gs[0,1])
# ax.pcolormesh(grd.lon_rho,grd.lat_rho,strain_norm.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),cmap='RdBu_r',vmin=-4,vmax=4)
# ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
# ax.set_xlim((153,156))

ax=fig.add_subplot(gs[0,1])
ax.pcolormesh(grd.lon_rho,grd.lat_rho,div_norm.isel(s_rho=-1).isel(ocean_time=summer_period_snapshot).squeeze(),cmap='RdBu_r',vmin=-3,vmax=3)
ax.contour(grd.lon_rho,grd.lat_rho,div_norm.isel(s_rho=-1).isel(ocean_time=summer_period_snapshot).squeeze(),colors='k',linewidths=.5)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.set_xlim((153,156))





plot some key features

In [ ]:
plt.cla()
plt.clf()
fig = plt.figure(figsize=[15,10])
ax = None

gs = gridspec.GridSpec(nrows=1,ncols=2,wspace=0.035, hspace=0.035)

ax=fig.add_subplot(gs[0,0],projection=ccrs.PlateCarree())
ax.pcolormesh(grd.lon_rho,grd.lat_rho,rel_vort.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),vmin=-1e-4,vmax=1e-4)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.set_extent([151.5, 156.1, -30, -24])


central_lat = -27  # adjust for your map
scale_length_km = 30
deg_per_km = 1 / (111.32 * np.cos(np.radians(central_lat)))
scale_length_deg = scale_length_km * deg_per_km
lon0 = 154.5  # scale bar position
lat0 = -27
# Add scale bar
ax.plot([lon0, lon0 + scale_length_deg], [lat0, lat0],
        transform=ccrs.PlateCarree(), color='black', lw=2)
ax.text(lon0 + scale_length_deg / 2, lat0 + 0.002*lat0, '30 km',
        transform=ccrs.PlateCarree(), ha='center', va='top', fontsize=9)

gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False


ax=fig.add_subplot(gs[0,1],projection=ccrs.PlateCarree())
ax.pcolormesh(grd.lon_rho,grd.lat_rho,ds_CTRL.temp.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),vmin=15,vmax=25,cmap='cmo.thermal')
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.set_extent([151.5, 156.1, -30, -24])


central_lat = -27  # adjust for your map
scale_length_km = 30
deg_per_km = 1 / (111.32 * np.cos(np.radians(central_lat)))
scale_length_deg = scale_length_km * deg_per_km
lon0 = 154.5  # scale bar position
lat0 = -27
# Add scale bar
ax.plot([lon0, lon0 + scale_length_deg], [lat0, lat0],
        transform=ccrs.PlateCarree(), color='black', lw=2)
ax.text(lon0 + scale_length_deg / 2, lat0 + 0.002*lat0, '30 km',
        transform=ccrs.PlateCarree(), ha='center', va='top', fontsize=9)

gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False

In [ ]:
image_save_dpi = 1000

plt.rcParams['figure.dpi'] = image_save_dpi
plt.rcParams.update({'font.size': 16})
plt.rcParams['xtick.labelsize']=16

In [ ]:
plt.cla()
plt.clf()
fig = plt.figure(figsize=[15,10])
ax = None

gs = gridspec.GridSpec(nrows=1,ncols=1,wspace=0.035, hspace=0.035)

ax=fig.add_subplot(gs[0,0],projection=ccrs.PlateCarree())
im=ax.pcolormesh(grd.lon_rho,grd.lat_rho,ds_CTRL.temp.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),cmap='cmo.thermal',vmin=14,vmax=21)
# ax.contour(grd.lon_rho,grd.lat_rho,ds_CTRL.temp.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),levels=np.arange(15,30,1),colors='k',linewidths=0.5)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.set_extent([153, 156.25, -28.75, -26.5])


central_lat = -27.5  # adjust for your map
scale_length_km = 30
deg_per_km = 1 / (111.32 * np.cos(np.radians(central_lat)))
scale_length_deg = scale_length_km * deg_per_km
lon0 = 153.1  # scale bar position
lat0 = -28.6
# Add scale bar
ax.plot([lon0, lon0 + scale_length_deg], [lat0, lat0],
        transform=ccrs.PlateCarree(), color='black', lw=4)
ax.text(lon0 + scale_length_deg / 2, lat0 + 0.002*lat0, '30 km',
        transform=ccrs.PlateCarree(), ha='center', va='top', fontsize=9)

gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False


#add colourbar
cax = inset_axes(ax,
                width="3%",  # width = 10% of parent_bbox width
                height="30%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.01,.2, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax) 
cbar.set_label('Temperature($^\circ$C)')

plt the mke and eke

In [ ]:


plt.cla()
plt.clf()
fig = plt.figure(figsize=[15,10])
ax = None

gs = gridspec.GridSpec(nrows=1,ncols=2,wspace=0.035, hspace=0.035)

choose_depth=0

ax=fig.add_subplot(gs[0,0],projection=ccrs.PlateCarree())
im = ax.pcolormesh(grd.lon_rho,grd.lat_rho,mke.isel(s_rho=-1),cmap='inferno')#,vmin=-3,vmax=3)
# ax.contour(grd.lon_psi,grd.lat_psi,rel_vort_norm.isel(s_rho=-1),colors='k',linewidths=.5)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')


cax = inset_axes(ax,
                width="2%",  # width = 10% of parent_bbox width
                height="10%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.01,0.1, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='vertical') 
cax.set_title('     MKE\n     (J/m²)')#,fontsize=14)

gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False

ax.set_extent([152.5, 158.81, -31.1, -24])


us_rho_s1   = ds_CTRL.u_eastward.mean(dim='ocean_time')
vs_rho_s1   = ds_CTRL.v_northward.mean(dim='ocean_time')
res=5
choose_depth=-1

ax=fig.add_subplot(gs[0,1],projection=ccrs.PlateCarree())
im=ax.pcolormesh(grd.lon_rho,grd.lat_rho,ds_CTRL.zeta.mean(dim='ocean_time'),cmap='cmo.amp',vmin=0,vmax=1)


cax = inset_axes(ax,
                width="2%",  # width = 10% of parent_bbox width
                height="10%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.01,0.1, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='vertical') 
cax.set_ylabel(r' mean SSH (m)')#,fontsize=14)

# ax.contour(grd.lon_rho,grd.lat_rho,div_norm.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),colors='k',linewidths=.5)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
# ax.quiver(ds_CTRL.lon_rho.values[0::res,0::res], ds_CTRL.lat_rho.values[0::res,0::res], 
#           us_rho_s1.isel(s_rho=choose_depth).squeeze().values[0::res,0::res], vs_rho_s1.isel(s_rho=choose_depth).squeeze().values[0::res,0::res],
#           scale=10, width=0.001,alpha=0.5)

snap_shot = ds_CTRL.mean(dim='ocean_time')
snap_shot["umag"] = np.sqrt(snap_shot.u_eastward.isel(s_rho=-1).squeeze()**2+snap_shot.v_northward.isel(s_rho=-1).squeeze()**2)

# other streamplot options:
str_kwargs = {"color":snap_shot.umag.values,
              "linewidth":1,
              "arrowsize":1,
              "density":4,
              "cmap":"pink",
             "transform":ccrs.PlateCarree()}
st = ax.streamplot(snap_shot.lon_rho.values, snap_shot.lat_rho.values, snap_shot.u_eastward.isel(s_rho=-1).values, snap_shot.v_northward.isel(s_rho=-1).values,**str_kwargs)

gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False
gl.left_labels = False

ax.set_extent([152.5, 158.81, -31.1, -24])

# ax.set_extent([153, 156, -29, -25])

# ax.set_xlim((153,156))

poster SMS exploration plots

In [ ]:


plt.cla()
plt.clf()
fig = plt.figure(figsize=[20,24])
ax = None

gs = gridspec.GridSpec(nrows=2,ncols=2,wspace=0.05, hspace=0.05)

choose_depth=0


ax=fig.add_subplot(gs[0,0],projection=ccrs.PlateCarree())
im = ax.pcolormesh(grd.lon_rho,grd.lat_rho,ds_CTRL.temp.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),cmap='cmo.thermal',vmin=15,vmax=22)
us_rho_s1   = ds_CTRL.u_eastward.isel(ocean_time=winter_period_snapshot)
vs_rho_s1   = ds_CTRL.v_northward.isel(ocean_time=winter_period_snapshot)
res=5
choose_depth=-1
ax.quiver(ds_CTRL.lon_rho.values[0::res,0::res], ds_CTRL.lat_rho.values[0::res,0::res], 
          us_rho_s1.isel(s_rho=choose_depth).squeeze().values[0::res,0::res], vs_rho_s1.isel(s_rho=choose_depth).squeeze().values[0::res,0::res],
          scale=65, width=0.001,alpha=0.5)
# ax.contour(grd.lon_psi,grd.lat_psi,rel_vort_norm.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),colors='k',linewidths=.5)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.set_extent([152.5, 158.81, -31.1, -24])


cax = inset_axes(ax,
                width="1%",  # width = 10% of parent_bbox width
                height="25%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.01,0.2, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='vertical') 
cax.set_ylabel(r'SST ($\circ$C)')#,fontsize=14)



central_lat = -27.5  # adjust for your map
scale_length_km = 30
deg_per_km = 1 / (111.32 * np.cos(np.radians(central_lat)))
scale_length_deg = scale_length_km * deg_per_km
lon0 = 152.6  # scale bar position
lat0 = -30.3
# Add scale bar
ax.plot([lon0, lon0 + scale_length_deg], [lat0, lat0],
        transform=ccrs.PlateCarree(), color='black', lw=4)
ax.text(lon0 + scale_length_deg / 2, lat0 + 0.002*lat0, '30 km',
        transform=ccrs.PlateCarree(), ha='center', va='top', fontsize=9)

ax.quiver(152.6, -30, 
          1, 0,
          scale=65, width=0.001,alpha=1)
ax.text(152.78,-30,'1 m/s',va='center')


gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False



ax=fig.add_subplot(gs[0,1],projection=ccrs.PlateCarree())
im = ax.pcolormesh(grd.lon_rho,grd.lat_rho,rel_vort_norm.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),cmap='RdBu_r',vmin=-3,vmax=3)
# ax.contour(grd.lon_psi,grd.lat_psi,rel_vort_norm.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),colors='k',linewidths=.5)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.set_extent([152.5, 158.81, -31.1, -24])


cax = inset_axes(ax,
                width="1%",  # width = 10% of parent_bbox width
                height="25%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.01,0.2, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='vertical') 
cax.set_ylabel(r'Relative vorticity (normalised)')#,fontsize=14)



gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False
gl.left_labels = False




ax=fig.add_subplot(gs[1,0],projection=ccrs.PlateCarree())
im = ax.pcolormesh(grd.lon_rho,grd.lat_rho,div_norm.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),cmap='RdBu_r',vmin=-1.5,vmax=1.5)
# ax.contour(grd.lon_rho,grd.lat_rho,div_norm.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),colors='k',linewidths=.5)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.set_extent([152.5, 158.81, -31.1, -24])

cax = inset_axes(ax,
                width="1%",  # width = 10% of parent_bbox width
                height="25%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.01,0.2, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='vertical') 
cax.set_ylabel(r'Horizontal convergence (normalised)')#,fontsize=14)



gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False



ax=fig.add_subplot(gs[1,1],projection=ccrs.PlateCarree())
im = ax.pcolormesh(grd.lon_psi,grd.lat_psi,zeta_skew,cmap='RdBu_r',vmin=-5,vmax=5)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.set_extent([152.5, 158.81, -31.1, -24])

cax = inset_axes(ax,
                width="1%",  # width = 10% of parent_bbox width
                height="25%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.01,0.2, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='vertical') 
cax.set_ylabel(r'Skewness of relative vorticity')#,fontsize=14)


gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False
gl.left_labels = False





In [ ]:
fig = plt.figure(figsize=[20,24])

gs = gridspec.GridSpec(nrows=2,ncols=2,wspace=0.05, hspace=0.05)


ax=fig.add_subplot(gs[1,1],projection=ccrs.PlateCarree())
im = ax.pcolormesh(grd.lon_psi,grd.lat_psi,zeta_skew,cmap='RdBu_r',vmin=-5,vmax=5)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.set_extent([152.5, 158.81, -31.1, -24])

cax = inset_axes(ax,
                width="1%",  # width = 10% of parent_bbox width
                height="25%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.01,0.2, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='vertical') 
cax.set_ylabel(r'Skewness of relative vorticity')#,fontsize=14)

background plots

In [ ]:
# load for nice cartopy plots

import cartopy.io.img_tiles as cimgt
Coast = cfeature.NaturalEarthFeature(category='physical',scale='50m',facecolor='none', name='coastline')
google_terrain = cimgt.GoogleTiles(style="satellite")

In [ ]:
# Background plot

us_rho_s1   = ds_CTRL.u_eastward.isel(ocean_time=winter_period_snapshot)
vs_rho_s1   = ds_CTRL.v_northward.isel(ocean_time=winter_period_snapshot)

res=3
choose_depth=-2


plt.cla()
plt.clf()
fig = plt.figure(figsize=[33*1.3,46*1.3])
ax = None
gs = gridspec.GridSpec(nrows=1,ncols=1,wspace=0.035, hspace=0.035)


ax=fig.add_subplot(gs[0,0],projection=ccrs.PlateCarree())
ax.add_image(google_terrain, 10, alpha=0.8)
ax.add_feature(Coast, edgecolor='black')
im2 = (ds_CTRL.mask_rho*(ds_CTRL.mask_rhoNaN)*ds_CTRL.temp.isel(s_rho=choose_depth).isel(ocean_time=winter_period_snapshot).squeeze()).plot.pcolormesh(
          ax=ax,x='lon_rho',y='lat_rho',cmap='cmo.thermal',add_colorbar=False,vmin=15,vmax=22)
ax.quiver(ds_CTRL.lon_rho.values[0::res,0::res], ds_CTRL.lat_rho.values[0::res,0::res], 
          us_rho_s1.isel(s_rho=choose_depth).squeeze().values[0::res,0::res], vs_rho_s1.isel(s_rho=choose_depth).squeeze().values[0::res,0::res],
          scale=40, width=0.001,alpha=0.5)


# co2 = (ds_CTRL.h*ds_CTRL.mask_rho).plot.contour(ax=ax,x='x_rho',y='y_rho',levels=(0,250,500,750,1000,2000,3000),colors='k',linestyles='-',linewidths=1)
# co3 = ax.contour(ds_CTRL.x_rho,ds_CTRL.y_rho,ds_CTRL.zice*ds_CTRL.mask_rho,levels=(-1,0),colors='C1',linestyles='-',linewidths=1)

ax.set_extent([151.5, 156.1, -30, -24])
# ax.set_ylabel('Northings (km)')
# ax.set_xlabel('Eastings (km)')
# # ax.grid()
# scale_ticks = 1e3
# ticks_x = mticker.FuncFormatter(lambda x, pos: '{0:g}'.format(x/scale_ticks))
# ax.xaxis.set_major_formatter(ticks_x)
# ticks_y = mticker.FuncFormatter(lambda x, pos: '{0:g}'.format(x/scale_ticks))
# ax.yaxis.set_major_formatter(ticks_y)
# ax.set_xticklabels([])
ax.set_xlabel('')
ax.set_xticklabels([])
ax.set_title('')
# ax.text(0.01, 0.99, 'a  Late summer (DOY 20 to 80)', transform=ax.transAxes,fontsize=14, fontweight='bold', va='top')



cax = inset_axes(ax,
                width="10%",  # width = 10% of parent_bbox width
                height="2%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.02,0.01, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im2, cax=cax, orientation='horizontal') 
cax.set_title(r'      Potential temperature ($^\circ$C)')#,fontsize=14)


# fig.suptitle('Bottom temperature and velocities')